<a href="https://colab.research.google.com/github/treborskrub/Fundamental-Cores/blob/main/copilotfullengn1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Any, Dict, List, Optional
import time
import math
import json

class CommandType(str, Enum):
    STEP = "step"
    REFLECT = "reflect"
    PAUSE = "pause"
    RESUME = "resume"
    JUMP = "jump"
    REPLAY = "replay"
    FILTER = "filter"
    SET = "set"
    EXPORT = "export"
    HELP = "help"
    MACRO = "macro"

@dataclass
class Command:
    type: CommandType
    target: Optional[str] = None
    args: Dict[str, Any] = field(default_factory=dict)
    request_id: Optional[str] = None
    actor: str = "dashboard"
    timestamp: float = field(default_factory=time.time)

@dataclass
class CommandResult:
    ok: bool
    message: str
    state: Optional[Dict[str, Any]] = None
    trace: List[Dict[str, Any]] = field(default_factory=list)

@dataclass
class ParseResult:
    ok: bool
    command: Optional[Command] = None
    error: Optional[str] = None

@dataclass
class Macro:
    name: str
    steps: List[str]
    description: str = ""
    created_at: float = field(default_factory=time.time)
    tags: List[str] = field(default_factory=list)
    enabled: bool = True

class MacroStore:
    def __init__(self):
        self.macros: Dict[str, Macro] = {}

    def save(self, name: str, steps: List[str], description: str = ""):
        self.macros[name] = Macro(name=name, steps=steps, description=description)

    def get(self, name: str):
        return self.macros.get(name)

    def delete(self, name: str):
        return self.macros.pop(name, None)

    def list(self):
        return list(self.macros.values())

class SimpleEngine:
    def __init__(self):
        self.clock = 0
        self.state = {"coherence": 0.5, "curiosity": 0.5, "arousal": 0.5, "valence": 0.0}
        self.snapshots = []
        self.last_step = None

    def _snapshot(self):
        snap = {"clock": self.clock, "state": dict(self.state)}
        self.snapshots.append(snap)
        return snap

    def step(self, tokens, anomalies=None):
        self.clock += 1
        vals = tokens if tokens else [0.4, 0.4, 0.4]
        self.state["coherence"] = max(0.0, min(1.0, sum(vals) / len(vals)))
        self.state["curiosity"] = max(0.0, min(1.0, self.state["curiosity"] + 0.01))
        self.last_step = {"tokens": tokens, "anomalies": anomalies}
        return self._snapshot()

    def reflect(self, intensity=1.0):
        self.state["valence"] = max(-1.0, min(1.0, self.state["valence"] + 0.05 * float(intensity)))
        return self._snapshot()

    def replay(self, index):
        if not self.snapshots:
            return {"error": "no snapshots"}
        index = max(0, min(int(index), len(self.snapshots) - 1))
        return self.snapshots[index]

_SAFE_FIELDS = {"coherence", "curiosity", "arousal", "valence"}

def parse_step_command(tokens):
    args = {}
    i = 1
    while i < len(tokens):
        t = tokens[i].lower()
        if t == "tokens":
            i += 1
            vals = []
            while i < len(tokens) and tokens[i].lower() != "anomalies":
                vals.append(float(tokens[i].rstrip(",")))
                i += 1
            args["tokens"] = vals
            continue
        if t == "anomalies":
            i += 1
            vals = []
            while i < len(tokens):
                vals.append(float(tokens[i].rstrip(",")))
                i += 1
            args["anomalies"] = vals
            continue
        i += 1
    if "tokens" not in args:
        if len(tokens) > 1:
            try:
                n = int(tokens[1])
                args["tokens"] = [0.4] * max(1, n)
            except ValueError:
                args["tokens"] = [0.4, 0.4, 0.4]
        else:
            args["tokens"] = [0.4, 0.4, 0.4]
    return Command(type=CommandType.STEP, args=args)

def parse_set_command(tokens):
    if len(tokens) < 5:
        return None, "set requires: set agent <id> <field> <value>"
    if tokens[1].lower() != "agent":
        return None, "only agent targets are supported"
    try:
        agent_id = int(tokens[2])
    except ValueError:
        return None, "agent id must be integer"
    field = tokens[3]
    if field not in _SAFE_FIELDS:
        return None, f"unsupported field: {field}"
    value = tokens[4]
    try:
        value = float(value)
    except ValueError:
        pass
    return Command(type=CommandType.SET, target=f"agent:{agent_id}", args={"field": field, "value": value}), None

def parse_filter_command(tokens):
    if len(tokens) < 3:
        return None, "filter requires: filter agent <id>"
    if tokens[1].lower() != "agent":
        return None, "only agent filters are supported"
    try:
        agent_id = int(tokens[2])
    except ValueError:
        return None, "agent id must be integer"
    return Command(type=CommandType.FILTER, args={"agent_id": agent_id}), None

def parse_macro_save(tokens):
    raw = " ".join(tokens)
    if "=" not in raw:
        return None, "macro save requires: macro save <name> = <cmd1> -> <cmd2>"
    left, right = raw.split("=", 1)
    left_tokens = left.strip().split()
    if len(left_tokens) < 3:
        return None, "macro save requires a name"
    name = left_tokens[2]
    steps = [s.strip() for s in right.split("->") if s.strip()]
    if not steps:
        return None, "macro must contain at least one step"
    return Command(type=CommandType.MACRO, args={"action": "save", "name": name, "steps": steps}), None

def parse_macro_run(tokens):
    if len(tokens) < 3:
        return None, "macro run requires a name"
    return Command(type=CommandType.MACRO, args={"action": "run", "name": tokens[2]}), None

def parse_macro_list(tokens):
    return Command(type=CommandType.MACRO, args={"action": "list"}), None

def parse_macro_show(tokens):
    if len(tokens) < 3:
        return None, "macro show requires a name"
    return Command(type=CommandType.MACRO, args={"action": "show", "name": tokens[2]}), None

def parse_command(text: str) -> ParseResult:
    tokens = text.strip().split()
    if not tokens:
        return ParseResult(ok=True, command=Command(type=CommandType.HELP))
    head = tokens[0].lower()
    try:
        if head == "pause":
            return ParseResult(ok=True, command=Command(type=CommandType.PAUSE))
        if head == "resume":
            return ParseResult(ok=True, command=Command(type=CommandType.RESUME))
        if head == "help":
            return ParseResult(ok=True, command=Command(type=CommandType.HELP))
        if head == "reflect":
            intensity = float(tokens[1]) if len(tokens) > 1 else 1.0
            return ParseResult(ok=True, command=Command(type=CommandType.REFLECT, args={"intensity": intensity}))
        if head == "jump":
            if len(tokens) < 2:
                return ParseResult(ok=False, error="jump requires index")
            return ParseResult(ok=True, command=Command(type=CommandType.JUMP, args={"index": int(tokens[1])}))
        if head == "replay":
            idx = int(tokens[1]) if len(tokens) > 1 else 0
            return ParseResult(ok=True, command=Command(type=CommandType.REPLAY, args={"index": idx}))
        if head == "step":
            return ParseResult(ok=True, command=parse_step_command(tokens))
        if head == "set":
            cmd, err = parse_set_command(tokens)
            if err:
                return ParseResult(ok=False, error=err)
            return ParseResult(ok=True, command=cmd)
        if head == "filter":
            cmd, err = parse_filter_command(tokens)
            if err:
                return ParseResult(ok=False, error=err)
            return ParseResult(ok=True, command=cmd)
        if head == "macro":
            if len(tokens) < 2:
                return ParseResult(ok=False, error="macro requires action")
            action = tokens[1].lower()
            if action == "save":
                cmd, err = parse_macro_save(tokens)
                return ParseResult(ok=True, command=cmd) if not err else ParseResult(ok=False, error=err)
            if action == "run":
                cmd, err = parse_macro_run(tokens)
                return ParseResult(ok=True, command=cmd) if not err else ParseResult(ok=False, error=err)
            if action == "list":
                cmd, err = parse_macro_list(tokens)
                return ParseResult(ok=True, command=cmd) if not err else ParseResult(ok=False, error=err)
            if action == "show":
                cmd, err = parse_macro_show(tokens)
                return ParseResult(ok=True, command=cmd) if not err else ParseResult(ok=False, error=err)
            return ParseResult(ok=False, error=f"unknown macro action: {action}")
        return ParseResult(ok=False, error=f"unknown command: {head}")
    except Exception as e:
        return ParseResult(ok=False, error=str(e))

class CommandPlane:
    def __init__(self, engine, store):
        self.engine = engine
        self.macros = store
        self.paused = False
        self.filters: Dict[str, Any] = {}
        self.trace: List[Dict[str, Any]] = []

    def _log(self, command: Command, ok: bool, message: str, data: Optional[Dict[str, Any]] = None):
        entry = {
            "timestamp": command.timestamp,
            "type": command.type.value,
            "actor": command.actor,
            "target": command.target,
            "args": command.args,
            "ok": ok,
            "message": message,
        }
        if data is not None:
            entry["data"] = data
        self.trace.append(entry)
        self.trace[:] = self.trace[-500:]
        return entry

    def validate(self, command: Command):
        t = command.type
        a = command.args
        if t == CommandType.STEP and "tokens" not in a:
            return False, "step requires tokens"
        if t == CommandType.REFLECT and "intensity" in a and not isinstance(a["intensity"], (int, float)):
            return False, "intensity must be numeric"
        if t == CommandType.JUMP and "index" not in a:
            return False, "jump requires index"
        if t == CommandType.SET and ("field" not in a or "value" not in a):
            return False, "set requires field and value"
        return True, "ok"

    def execute(self, command: Command) -> CommandResult:
        ok, msg = self.validate(command)
        if not ok:
            entry = self._log(command, False, msg)
            return CommandResult(False, msg, state=self.state(), trace=[entry])

        t = command.type
        if t == CommandType.PAUSE:
            self.paused = True
            entry = self._log(command, True, "paused")
            return CommandResult(True, "paused", state=self.state(), trace=[entry])

        if t == CommandType.RESUME:
            self.paused = False
            entry = self._log(command, True, "resumed")
            return CommandResult(True, "resumed", state=self.state(), trace=[entry])

        if t == CommandType.STEP:
            if self.paused:
                entry = self._log(command, False, "engine paused")
                return CommandResult(False, "engine paused", state=self.state(), trace=[entry])
            out = self.engine.step(command.args["tokens"], command.args.get("anomalies"))
            entry = self._log(command, True, "stepped", out)
            return CommandResult(True, "stepped", state=self.state(), trace=[entry])

        if t == CommandType.REFLECT:
            out = self.engine.reflect(float(command.args.get("intensity", 1.0)))
            entry = self._log(command, True, "reflected", out)
            return CommandResult(True, "reflected", state=self.state(), trace=[entry])

        if t == CommandType.JUMP:
            idx = int(command.args["index"])
            out = self.engine.replay(idx)
            entry = self._log(command, True, "replayed", out)
            return CommandResult(True, "replayed", state=self.state(), trace=[entry])

        if t == CommandType.REPLAY:
            idx = int(command.args.get("index", max(0, len(self.engine.snapshots) - 1)))
            out = self.engine.replay(idx)
            entry = self._log(command, True, "replayed", out)
            return CommandResult(True, "replayed", state=self.state(), trace=[entry])

        if t == CommandType.FILTER:
            self.filters = dict(command.args)
            entry = self._log(command, True, "filters updated")
            return CommandResult(True, "filters updated", state=self.state(), trace=[entry])

        if t == CommandType.SET:
            target = command.target or ""
            field = command.args["field"]
            value = command.args["value"]
            if target.startswith("agent:"):
                entry = self._log(command, True, f"set {target}.{field}", {"value": value})
                if field in self.engine.state:
                    self.engine.state[field] = value
                return CommandResult(True, f"set {target}.{field}", state=self.state(), trace=[entry])
            entry = self._log(command, False, "unsupported target")
            return CommandResult(False, "unsupported target", state=self.state(), trace=[entry])

        if t == CommandType.MACRO:
            action = command.args.get("action")
            if action == "save":
                name = command.args["name"]
                steps = command.args["steps"]
                self.macros.save(name, steps)
                entry = self._log(command, True, f"macro saved: {name}", {"steps": steps})
                return CommandResult(True, f"macro saved: {name}", state=self.state(), trace=[entry])

            if action == "list":
                items = [{
                    "name": m.name,
                    "enabled": m.enabled,
                    "count": len(m.steps)
                } for m in self.macros.list()]
                entry = self._log(command, True, "macro list")
                return CommandResult(True, "macro list", state=self.state(), trace=[entry])

            if action == "show":
                name = command.args["name"]
                macro = self.macros.get(name)
                if not macro:
                    entry = self._log(command, False, "macro not found")
                    return CommandResult(False, "macro not found", state=self.state(), trace=[entry])
                entry = self._log(command, True, f"macro show: {name}", {"steps": macro.steps})
                return CommandResult(True, f"macro show: {name}", state=self.state(), trace=[entry])

            if action == "run":
                name = command.args["name"]
                macro = self.macros.get(name)
                if not macro or not macro.enabled:
                    entry = self._log(command, False, "macro not found")
                    return CommandResult(False, "macro not found", state=self.state(), trace=[entry])

                expanded = []
                for raw in macro.steps:
                    parsed = parse_command(raw)
                    if not parsed.ok:
                        entry = self._log(command, False, f"macro parse failed: {raw}")
                        return CommandResult(False, f"macro parse failed: {raw}", state=self.state(), trace=[entry])
                    res = self.execute(parsed.command)
                    expanded.append({"raw": raw, "result": res.message, "ok": res.ok})
                    if not res.ok:
                        entry = self._log(command, False, f"macro aborted at: {raw}", {"expanded": expanded})
                        return CommandResult(False, f"macro aborted at: {raw}", state=self.state(), trace=[entry])

                entry = self._log(command, True, f"macro complete: {name}", {"expanded": expanded})
                return CommandResult(True, f"macro complete: {name}", state=self.state(), trace=[entry])

            entry = self._log(command, False, "unknown macro action")
            return CommandResult(False, "unknown macro action", state=self.state(), trace=[entry])

        if t == CommandType.EXPORT:
            entry = self._log(command, True, "exported")
            return CommandResult(True, "exported", state=self.state(), trace=[entry])

        if t == CommandType.HELP:
            entry = self._log(command, True, "help")
            return CommandResult(True, "help", state=self.state(), trace=[entry])

        entry = self._log(command, False, "unknown command")
        return CommandResult(False, "unknown command", state=self.state(), trace=[entry])

    def state(self):
        return {
            "paused": self.paused,
            "filters": self.filters,
            "clock": getattr(self.engine, "clock", None),
            "trace_len": len(self.trace),
            "macro_count": len(self.macros.macros),
        }

def run_text(plane, text):
    parsed = parse_command(text)
    if not parsed.ok:
        return {"ok": False, "message": parsed.error}
    result = plane.execute(parsed.command)
    return asdict(result)

engine = SimpleEngine()
store = MacroStore()
plane = CommandPlane(engine, store)

tests = [
    "help",
    "step tokens 0.4 0.5 0.6",
    "reflect 1.2",
    "set agent 1 coherence 0.82",
    "macro save stabilize = pause -> reflect 1.0 -> resume",
    "macro list",
    "macro show stabilize",
    "macro run stabilize",
    "pause",
    "step tokens 0.1 0.2 0.3",
    "resume",
    "step tokens 0.1 0.2 0.3",
]

for t in tests:
    out = run_text(plane, t)
    print(f"\nCOMMAND: {t}")
    print(json.dumps(out, indent=2, default=str))


COMMAND: help
{
  "ok": true,
  "message": "help",
  "state": {
    "paused": false,
    "filters": {},
    "clock": 0,
    "trace_len": 1,
    "macro_count": 0
  },
  "trace": [
    {
      "timestamp": 1782230490.4533758,
      "type": "help",
      "actor": "dashboard",
      "target": null,
      "args": {},
      "ok": true,
      "message": "help"
    }
  ]
}

COMMAND: step tokens 0.4 0.5 0.6
{
  "ok": true,
  "message": "stepped",
  "state": {
    "paused": false,
    "filters": {},
    "clock": 1,
    "trace_len": 2,
    "macro_count": 0
  },
  "trace": [
    {
      "timestamp": 1782230490.453634,
      "type": "step",
      "actor": "dashboard",
      "target": null,
      "args": {
        "tokens": [
          0.4,
          0.5,
          0.6
        ]
      },
      "ok": true,
      "message": "stepped",
      "data": {
        "clock": 1,
        "state": {
          "coherence": 0.5,
          "curiosity": 0.51,
          "arousal": 0.5,
          "valence": 0.0
     